In [ ]:
import os
import random
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf
import tensorflow_datasets as tfds

from tensorflow.keras import layers, models
from tensorflow.keras.applications import ResNet50
from sklearn.model_selection import train_test_split
from tensorflow.keras.applications.resnet50 import preprocess_input
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report
)

os.makedirs("Figures", exist_ok=True)

In [ ]:
SEED = 42

os.environ["PYTHONHASHSEED"] = str(SEED)

random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

IMG_SIZE = 224
NUM_CLASSES = 5
BATCH_SIZE = 32

EPOCHS_FROZEN = 10
EPOCHS_FINE = 5

AUTOTUNE = tf.data.AUTOTUNE

In [ ]:
(ds, ds_info) = tfds.load(
    "tf_flowers",
    split="train",
    as_supervised=True,
    with_info=True
)

class_names = ds_info.features["label"].names

print(class_names)
print("Total images:", ds_info.splits["train"].num_examples)

In [ ]:
images = []
labels = []

for image, label in tfds.as_numpy(ds):
    image = tf.image.resize(image, (IMG_SIZE, IMG_SIZE)).numpy()
    image = image.astype(np.uint8)

    images.append(image)
    labels.append(label)

images = np.stack(images)
labels = np.array(labels)

print("Images shape:", images.shape)
print("Labels shape:", labels.shape)

In [ ]:
indices = np.arange(len(labels))

train_idx, temp_idx = train_test_split(
    indices,
    test_size=0.30,
    stratify=labels,
    random_state=SEED
)

val_idx, test_idx = train_test_split(
    temp_idx,
    test_size=0.50,
    stratify=labels[temp_idx],
    random_state=SEED
)

print("Training:", len(train_idx))
print("Validation:", len(val_idx))
print("Testing:", len(test_idx))

np.savez(
    "tf_flowers_split.npz",
    train_idx=train_idx,
    val_idx=val_idx,
    test_idx=test_idx
)

In [ ]:
def preprocess_image(image, label):
    image = tf.image.resize(image, (IMG_SIZE, IMG_SIZE))
    image = tf.cast(image, tf.float32)
    image = preprocess_input(image)
    return image, label


augmentation = tf.keras.Sequential([
    layers.Input(shape=(IMG_SIZE, IMG_SIZE, 3)),
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.05),
    layers.RandomZoom(0.10),
    layers.RandomTranslation(0.05, 0.05),
    layers.RandomContrast(0.10)
])

In [ ]:
def create_dataset(indices, training=False):
    x = images[indices]
    y = labels[indices]

    ds = tf.data.Dataset.from_tensor_slices((x, y))

    if training:
        ds = ds.shuffle(
            buffer_size=len(indices),
            seed=SEED,
            reshuffle_each_iteration=True
        )

    ds = ds.batch(BATCH_SIZE)

    if training:
        ds = ds.map(
            lambda x, y: (augmentation(tf.cast(x, tf.float32), training=True), y),
            num_parallel_calls=AUTOTUNE
        )

    ds = ds.map(
        lambda x, y: (preprocess_input(tf.cast(x, tf.float32)), y),
        num_parallel_calls=AUTOTUNE
    )

    ds = ds.prefetch(AUTOTUNE)

    return ds

In [ ]:
train_ds = create_dataset(train_idx, training=True)
val_ds = create_dataset(val_idx, training=False)
test_ds = create_dataset(test_idx, training=False)

images_batch, labels_batch = next(iter(train_ds))

print("Image batch shape:", images_batch.shape)
print("Label batch shape:", labels_batch.shape)

In [ ]:
def display_image(image):
    image = image.numpy()
    image = image + [103.939, 116.779, 123.68]
    image = image[..., ::-1]
    image = np.clip(image / 255.0, 0, 1)
    return image

fig = plt.figure(figsize=(12, 8))

for i in range(10):
    image = images[train_idx[i]]
    label = labels[train_idx[i]]

    image = tf.image.resize(image, (IMG_SIZE, IMG_SIZE))
    image = tf.cast(image, tf.float32)
    image = preprocess_input(image)

    plt.subplot(2, 5, i + 1)
    plt.imshow(display_image(image))
    plt.title(class_names[label])
    plt.axis("off")

plt.tight_layout()
fig.savefig(
    "Figures/sample_flower_images.pdf",
    format="pdf",
    dpi=600,
    bbox_inches="tight"
)
plt.show()

In [ ]:
os.makedirs("Figures", exist_ok=True)

def class_distribution(indices):
    return np.bincount(labels[indices], minlength=NUM_CLASSES)

splits = {
    "Training": class_distribution(train_idx),
    "Validation": class_distribution(val_idx),
    "Testing": class_distribution(test_idx)
}

fig, ax = plt.subplots(figsize=(10, 6))
x = np.arange(NUM_CLASSES)
width = 0.25

for i, (name, counts) in enumerate(splits.items()):
    ax.bar(x + (i - 1) * width, counts, width, label=name)

ax.set_xticks(x)
ax.set_xticklabels(class_names)
ax.set_xlabel("Flower Class")
ax.set_ylabel("Number of Images")
ax.set_title("Class Distribution Across Dataset Splits")
ax.legend()

plt.tight_layout()
fig.savefig("Figures/class_distribution.pdf", format="pdf", dpi=600, bbox_inches="tight")
plt.show()
plt.close(fig)

In [ ]:
base_model = ResNet50(
    weights="imagenet",
    include_top=False,
    input_shape=(IMG_SIZE, IMG_SIZE, 3)
)

base_model.trainable = False

inputs = layers.Input(shape=(IMG_SIZE, IMG_SIZE, 3))
x = base_model(inputs, training=False)
x = layers.GlobalAveragePooling2D()(x)
x = layers.Dropout(0.3)(x)
outputs = layers.Dense(NUM_CLASSES, activation="softmax")(x)

baseline_model = models.Model(inputs, outputs, name="ResNet50_Baseline")

baseline_model.summary()

In [ ]:
baseline_model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

In [ ]:
os.makedirs("Figures", exist_ok=True)
os.makedirs("Models", exist_ok=True)
os.makedirs("Results", exist_ok=True)

In [ ]:
callbacks = [
    tf.keras.callbacks.ModelCheckpoint(
        "Models/baseline_best.keras",
        monitor="val_accuracy",
        save_best_only=True,
        mode="max"
    ),
    tf.keras.callbacks.EarlyStopping(
        monitor="val_loss",
        patience=3,
        restore_best_weights=True
    )
]

history_frozen = baseline_model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS_FROZEN,
    callbacks=callbacks
)

In [ ]:
base_model.trainable = True

for layer in base_model.layers:
    layer.trainable = layer.name.startswith("conv5_")
    if isinstance(layer, layers.BatchNormalization):
        layer.trainable = False

baseline_model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-5),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

history_fine = baseline_model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS_FINE,
    callbacks=callbacks
)

In [ ]:
baseline_model.save("Models/baseline_resnet50.keras")

def combine_history(key):
    return (
        history_frozen.history[key]
        + history_fine.history[key]
    )

epochs = range(1, len(combine_history("accuracy")) + 1)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(epochs, combine_history("accuracy"), label="Training")
axes[0].plot(epochs, combine_history("val_accuracy"), label="Validation")
axes[0].set_title("Baseline ResNet50 Accuracy")
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("Accuracy")
axes[0].legend()

axes[1].plot(epochs, combine_history("loss"), label="Training")
axes[1].plot(epochs, combine_history("val_loss"), label="Validation")
axes[1].set_title("Baseline ResNet50 Loss")
axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("Loss")
axes[1].legend()

fig.tight_layout()
fig.savefig(
    "Figures/baseline_training_curves.pdf",
    format="pdf",
    dpi=600,
    bbox_inches="tight"
)
plt.show()
plt.close(fig)

In [ ]:
baseline_model = tf.keras.models.load_model(
    "Models/baseline_resnet50.keras"
)

test_loss, test_accuracy = baseline_model.evaluate(test_ds, verbose=1)

print(f"Test Loss: {test_loss:.4f}")
print(f"Test Accuracy: {test_accuracy:.4f}")

In [ ]:
BASE_DIR = "/kaggle/working"
FIGURES_DIR = os.path.join(BASE_DIR, "Figures")
MODELS_DIR = os.path.join(BASE_DIR, "Models")
RESULTS_DIR = os.path.join(BASE_DIR, "Results")

for folder in [FIGURES_DIR, MODELS_DIR, RESULTS_DIR]:
    os.makedirs(folder, exist_ok=True)

In [ ]:
baseline_model = tf.keras.models.load_model(
    os.path.join(MODELS_DIR, "baseline_resnet50.keras")
)

y_true = []
y_pred = []
inference_times = []

for batch_images, batch_labels in test_ds:
    baseline_model(batch_images[:1], training=False)

    start = time.perf_counter()
    predictions = baseline_model(batch_images, training=False)
    predictions.numpy()
    elapsed = time.perf_counter() - start

    y_true.extend(batch_labels.numpy())
    y_pred.extend(np.argmax(predictions.numpy(), axis=1))
    inference_times.extend([elapsed / len(batch_images)] * len(batch_images))

baseline_results = {
    "Model": "Baseline ResNet50",
    "Position": "None",
    "Accuracy": accuracy_score(y_true, y_pred),
    "Macro Precision": precision_score(y_true, y_pred, average="macro", zero_division=0),
    "Macro Recall": recall_score(y_true, y_pred, average="macro", zero_division=0),
    "Macro F1": f1_score(y_true, y_pred, average="macro", zero_division=0),
    "Trainable Parameters": sum(
        np.prod(v.shape) for v in baseline_model.trainable_weights
    ),
    "Total Parameters": baseline_model.count_params(),
    "Inference Time (ms/image)": np.mean(inference_times) * 1000
}

print(pd.DataFrame([baseline_results]).T)

In [ ]:
baseline_df = pd.DataFrame([baseline_results])
baseline_df.to_csv(
    os.path.join(RESULTS_DIR, "baseline_metrics.csv"),
    index=False
)

In [ ]:
cm = confusion_matrix(y_true, y_pred, labels=np.arange(NUM_CLASSES))

fig, ax = plt.subplots(figsize=(8, 7))
im = ax.imshow(cm, interpolation="nearest")
fig.colorbar(im, ax=ax)

ax.set(
    xticks=np.arange(NUM_CLASSES),
    yticks=np.arange(NUM_CLASSES),
    xticklabels=class_names,
    yticklabels=class_names,
    xlabel="Predicted Class",
    ylabel="True Class",
    title="Baseline ResNet50 Confusion Matrix"
)

plt.setp(ax.get_xticklabels(), rotation=35, ha="right")

for i in range(NUM_CLASSES):
    for j in range(NUM_CLASSES):
        ax.text(j, i, str(cm[i, j]), ha="center", va="center")

fig.tight_layout()
fig.savefig(
    os.path.join(FIGURES_DIR, "baseline_confusion_matrix.pdf"),
    format="pdf",
    dpi=600,
    bbox_inches="tight"
)
plt.show()
plt.close(fig)

In [ ]:
@tf.keras.utils.register_keras_serializable()
class LightweightSelfAttention(layers.Layer):
    def build(self, input_shape):
        channels = int(input_shape[-1])
        key_dim = max(channels // 8, 1)

        self.query = layers.Conv2D(key_dim, 1)
        self.key = layers.Conv2D(key_dim, 1)
        self.value = layers.Conv2D(channels, 1)
        self.gamma = self.add_weight(
            name="gamma",
            shape=(),
            initializer="zeros",
            trainable=True
        )

    def call(self, x):
        shape = tf.shape(x)
        batch, height, width = shape[0], shape[1], shape[2]
        channels = x.shape[-1]

        q = tf.reshape(self.query(x), [batch, height * width, -1])
        k = tf.reshape(self.key(x), [batch, height * width, -1])
        v = tf.reshape(self.value(x), [batch, height * width, channels])

        scores = tf.matmul(q, k, transpose_b=True)
        scores = scores / tf.sqrt(tf.cast(tf.shape(q)[-1], tf.float32))
        weights = tf.nn.softmax(scores, axis=-1)

        attended = tf.matmul(weights, v)
        attended = tf.reshape(attended, [batch, height, width, channels])

        return x + self.gamma * attended


def channel_attention(x, reduction=16):
    channels = int(x.shape[-1])
    hidden = layers.GlobalAveragePooling2D(keepdims=True)(x)
    hidden = layers.Dense(max(channels // reduction, 1), activation="relu")(hidden)
    weights = layers.Dense(channels, activation="sigmoid")(hidden)
    return layers.Multiply()([x, weights])


@tf.keras.utils.register_keras_serializable()
class SpatialAttentionPooling(layers.Layer):
    def call(self, x):
        avg = tf.reduce_mean(x, axis=-1, keepdims=True)
        maximum = tf.reduce_max(x, axis=-1, keepdims=True)
        return tf.concat([avg, maximum], axis=-1)


def spatial_attention(x, kernel_size=7):
    pooled = SpatialAttentionPooling()(x)
    weights = layers.Conv2D(
        1, kernel_size, padding="same", activation="sigmoid"
    )(pooled)
    return layers.Multiply()([x, weights])

In [ ]:
def build_p3_attention_model(attention_type="channel", kernel_size=7):
    backbone = ResNet50(
        weights="imagenet",
        include_top=False,
        input_shape=(IMG_SIZE, IMG_SIZE, 3)
    )

    p3_tensor = backbone.get_layer("conv4_block6_out").output

    stem = models.Model(backbone.input, p3_tensor)
    tail = models.Model(p3_tensor, backbone.output)

    inputs = layers.Input(shape=(IMG_SIZE, IMG_SIZE, 3))
    x = stem(inputs)

    if attention_type == "channel":
        x = channel_attention(x)

    elif attention_type == "spatial":
        x = spatial_attention(x, kernel_size)

    elif attention_type == "channel_spatial":
        x = channel_attention(x)
        x = spatial_attention(x, kernel_size)

    elif attention_type == "spatial_channel":
        x = spatial_attention(x, kernel_size)
        x = channel_attention(x)

    elif attention_type == "self":
        x = LightweightSelfAttention()(x)

    elif attention_type == "combined":
        x = channel_attention(x)
        x = spatial_attention(x, kernel_size)
        x = LightweightSelfAttention()(x)

    else:
        raise ValueError(f"Unknown attention type: {attention_type}")

    x = tail(x)
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dropout(0.3)(x)
    outputs = layers.Dense(NUM_CLASSES, activation="softmax")(x)

    model = models.Model(
        inputs,
        outputs,
        name=f"ResNet50_P3_{attention_type}"
    )

    return model, stem, tail

In [ ]:
verification_model, verification_stem, _ = build_p3_attention_model("spatial")

sample = tf.zeros((1, IMG_SIZE, IMG_SIZE, 3))
before = verification_stem(sample, training=False)
after = spatial_attention(before)

print("Before attention:", before.shape)
print("After attention:", after.shape)
print("Classifier output:", verification_model(sample, training=False).shape)

del verification_model, verification_stem, sample, before, after
tf.keras.backend.clear_session()

In [ ]:
import json

def train_attention_model(attention_type, kernel_size=7):
    model, stem, tail = build_p3_attention_model(
        attention_type,
        kernel_size
    )

    stem.trainable = False
    tail.trainable = False

    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"]
    )

    model_path = os.path.join(
        MODELS_DIR, f"p3_{attention_type}_best.keras"
    )

    callbacks = [
        tf.keras.callbacks.ModelCheckpoint(
            model_path,
            monitor="val_accuracy",
            save_best_only=True,
            mode="max"
        ),
        tf.keras.callbacks.EarlyStopping(
            monitor="val_loss",
            patience=3,
            restore_best_weights=True
        )
    ]

    history_frozen = model.fit(
        train_ds,
        validation_data=val_ds,
        epochs=EPOCHS_FROZEN,
        callbacks=callbacks
    )

    stem.trainable = False
    tail.trainable = True

    for layer in tail.layers:
        layer.trainable = layer.name.startswith("conv5_")

        if isinstance(layer, layers.BatchNormalization):
            layer.trainable = False

    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=1e-5),
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"]
    )

    history_fine = model.fit(
        train_ds,
        validation_data=val_ds,
        epochs=EPOCHS_FINE,
        callbacks=callbacks
    )

    model.save(
    os.path.join(MODELS_DIR, f"p3_{attention_type}_final.keras")
)

    with open(
        os.path.join(RESULTS_DIR, f"p3_{attention_type}_history.json"),
        "w"
    ) as f:
        json.dump(
            {
                "frozen": history_frozen.history,
                "fine_tuning": history_fine.history
            },
            f
        )

    return model, history_frozen, history_fine

In [ ]:
channel_model, history_channel, history_channel_fine = train_attention_model(
    "channel"
)

In [ ]:
spatial_model, history_spatial, history_spatial_fine = train_attention_model(
    "spatial"
)

In [ ]:
for name in [
    "p3_channel_best.keras",
    "p3_channel_final.keras",
    "p3_spatial_best.keras",
    "p3_spatial_final.keras"
]:
    path = os.path.join(MODELS_DIR, name)
    print(f"{name}: {'Saved' if os.path.exists(path) else 'Missing'}")

for name in [
    "p3_channel_history.json",
    "p3_spatial_history.json"
]:
    path = os.path.join(RESULTS_DIR, name)
    print(f"{name}: {'Saved' if os.path.exists(path) else 'Missing'}")

In [ ]:
channel_spatial_model, history_channel_spatial, history_channel_spatial_fine = train_attention_model(
    "channel_spatial"
)

In [ ]:
self_attention_model, history_self, history_self_fine = train_attention_model(
    "self"
)

In [ ]:
combined_model, history_combined, history_combined_fine = train_attention_model(
    "combined"
)

In [ ]:
model_files = [
    "p3_channel_final.keras",
    "p3_spatial_final.keras",
    "p3_channel_spatial_final.keras",
    "p3_self_final.keras",
    "p3_combined_final.keras"
]

for name in model_files:
    path = os.path.join(MODELS_DIR, name)
    print(f"{name}: {'Saved' if os.path.exists(path) else 'Missing'}")

In [ ]:
feature_shapes = pd.DataFrame([
    {
        "Model": "A1 - Channel Attention",
        "Position": "P3",
        "Before Attention": "(14, 14, 1024)",
        "After Attention": "(14, 14, 1024)"
    },
    {
        "Model": "A2 - Spatial Attention",
        "Position": "P3",
        "Before Attention": "(14, 14, 1024)",
        "After Attention": "(14, 14, 1024)"
    },
    {
        "Model": "A3 - Channel + Spatial",
        "Position": "P3",
        "Before Attention": "(14, 14, 1024)",
        "After Attention": "(14, 14, 1024)"
    },
    {
        "Model": "A4 - Self-Attention",
        "Position": "P3",
        "Before Attention": "(14, 14, 1024)",
        "After Attention": "(14, 14, 1024)"
    },
    {
        "Model": "A5 - Combined Attention",
        "Position": "P3",
        "Before Attention": "(14, 14, 1024)",
        "After Attention": "(14, 14, 1024)"
    }
])

feature_shapes.to_csv(
    os.path.join(RESULTS_DIR, "attention_feature_dimensions.csv"),
    index=False
)

feature_shapes

In [ ]:
baseline_model = tf.keras.models.load_model(
    os.path.join(MODELS_DIR, "baseline_resnet50.keras")
)

models_to_evaluate = {
    "Baseline": baseline_model,
    "A1_Channel": channel_model,
    "A2_Spatial": spatial_model,
    "A3_Channel_Spatial": channel_spatial_model,
    "A4_Self_Attention": self_attention_model,
    "A5_Combined": combined_model
}

print("Models ready for evaluation:")
for name, model in models_to_evaluate.items():
    print(f"{name}: {model.output_shape}")

In [ ]:
y_true = np.concatenate([
    labels.numpy() for _, labels in test_ds
])

results = []
predictions = {}

for name, model in models_to_evaluate.items():
    y_prob = model.predict(test_ds, verbose=1)
    y_pred = np.argmax(y_prob, axis=1)
    predictions[name] = y_pred

    start = time.perf_counter()
    for images, _ in test_ds:
        model(images, training=False).numpy()
    elapsed = time.perf_counter() - start

    results.append({
        "Model": name,
        "Accuracy": accuracy_score(y_true, y_pred),
        "Macro Precision": precision_score(
            y_true, y_pred, average="macro", zero_division=0
        ),
        "Macro Recall": recall_score(
            y_true, y_pred, average="macro", zero_division=0
        ),
        "Macro F1": f1_score(
            y_true, y_pred, average="macro", zero_division=0
        ),
        "Total Parameters": model.count_params(),
        "Trainable Parameters": sum(
            int(tf.size(v)) for v in model.trainable_variables
        ),
        "Inference Time (ms/image)": elapsed / len(y_true) * 1000
    })

results_df = pd.DataFrame(results)
results_df.to_csv(
    os.path.join(RESULTS_DIR, "main_model_comparison.csv"),
    index=False
)

results_df.round(4)

In [ ]:
from sklearn.metrics import classification_report

class_names = getattr(
    test_ds, "class_names",
    ["daisy", "dandelion", "roses", "sunflowers", "tulips"]
)

for name, y_pred in predictions.items():
    report = classification_report(
        y_true,
        y_pred,
        labels=np.arange(NUM_CLASSES),
        target_names=class_names,
        zero_division=0,
        digits=4
    )

    with open(
        os.path.join(RESULTS_DIR, f"{name}_classification_report.txt"),
        "w"
    ) as f:
        f.write(report)

print("Saved comparison table and classification reports.")

In [ ]:
import matplotlib.pyplot as plt

metrics = [
    ("Accuracy", "Model Accuracy", "model_accuracy.pdf"),
    ("Macro F1", "Macro F1-Score", "model_macro_f1.pdf")
]

for metric, ylabel, filename in metrics:
    fig, ax = plt.subplots(figsize=(11, 6))
    ax.bar(results_df["Model"], results_df[metric])
    ax.set_ylabel(ylabel)
    ax.set_title(ylabel + " Comparison")
    ax.set_ylim(0, 1)
    ax.tick_params(axis="x", rotation=30)
    fig.tight_layout()
    fig.savefig(
        os.path.join(FIGURES_DIR, filename),
        format="pdf",
        dpi=600,
        bbox_inches="tight"
    )
    plt.show()
    plt.close(fig)

In [ ]:
def build_position_attention_model(position="P2", kernel_size=7):
    layer_names = {
        "P1": "conv2_block3_out",
        "P2": "conv3_block4_out",
        "P3": "conv4_block6_out",
        "P4": "conv5_block3_out"
    }

    if position not in layer_names:
        raise ValueError("Position must be P1, P2, P3, or P4")

    backbone = ResNet50(
        weights="imagenet",
        include_top=False,
        input_shape=(IMG_SIZE, IMG_SIZE, 3)
    )

    selected_tensor = backbone.get_layer(layer_names[position]).output

    stem = models.Model(backbone.input, selected_tensor)

    inputs = layers.Input(shape=(IMG_SIZE, IMG_SIZE, 3))
    x = stem(inputs)
    x = channel_attention(x)
    x = spatial_attention(x, kernel_size)

    if position != "P4":
        tail = models.Model(selected_tensor, backbone.output)
        x = tail(x)
    else:
        tail = None

    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dropout(0.3)(x)
    outputs = layers.Dense(NUM_CLASSES, activation="softmax")(x)

    model = models.Model(
        inputs,
        outputs,
        name=f"ResNet50_{position}_ChannelSpatial"
    )

    return model, stem, tail

In [ ]:
def train_position_model(position, kernel_size=7):
    model, stem, tail = build_position_attention_model(position, kernel_size)

    stem.trainable = False
    if tail is not None:
        tail.trainable = False

    model.compile(
        optimizer=tf.keras.optimizers.Adam(1e-3),
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"]
    )

    frozen_history = model.fit(
        train_ds,
        validation_data=val_ds,
        epochs=EPOCHS_FROZEN,
        callbacks=[
            tf.keras.callbacks.EarlyStopping(
                monitor="val_loss",
                patience=3,
                restore_best_weights=True
            )
        ]
    )

    if tail is not None:
        tail.trainable = True
        for layer in tail.layers:
            layer.trainable = layer.name.startswith("conv5_")
            if isinstance(layer, layers.BatchNormalization):
                layer.trainable = False

    model.compile(
        optimizer=tf.keras.optimizers.Adam(1e-5),
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"]
    )

    fine_history = model.fit(
        train_ds,
        validation_data=val_ds,
        epochs=EPOCHS_FINE,
        callbacks=[
            tf.keras.callbacks.EarlyStopping(
                monitor="val_loss",
                patience=3,
                restore_best_weights=True
            )
        ]
    )

    model.save(
        os.path.join(
            MODELS_DIR,
            f"{position}_channel_spatial_final.keras"
        )
    )

    with open(
        os.path.join(RESULTS_DIR, f"{position}_history.json"),
        "w"
    ) as f:
        json.dump({
            "frozen": frozen_history.history,
            "fine_tuning": fine_history.history
        }, f)

    return model, frozen_history, fine_history

In [ ]:
p1_model, p1_history, p1_history_fine = train_position_model("P1")

In [ ]:
p2_model, p2_history, p2_history_fine = train_position_model("P2")

In [ ]:
p4_model, p4_history, p4_history_fine = train_position_model("P4")

In [ ]:
position_models = {
    "P1": p1_model,
    "P2": p2_model,
    "P3": channel_spatial_model,
    "P4": p4_model
}

y_true = np.concatenate([y.numpy() for _, y in test_ds])
position_results = []

for position, model in position_models.items():
    probabilities = model.predict(test_ds, verbose=1)
    y_pred = np.argmax(probabilities, axis=1)

    start = time.perf_counter()
    for images, _ in test_ds:
        model(images, training=False).numpy()
    elapsed = time.perf_counter() - start

    position_results.append({
        "Position": position,
        "Accuracy": accuracy_score(y_true, y_pred),
        "Macro Precision": precision_score(y_true, y_pred, average="macro", zero_division=0),
        "Macro Recall": recall_score(y_true, y_pred, average="macro", zero_division=0),
        "Macro F1": f1_score(y_true, y_pred, average="macro", zero_division=0),
        "Total Parameters": model.count_params(),
        "Trainable Parameters": sum(int(tf.size(v)) for v in model.trainable_variables),
        "Inference Time (ms/image)": elapsed / len(y_true) * 1000
    })

position_df = pd.DataFrame(position_results)
position_df.to_csv(
    os.path.join(RESULTS_DIR, "position_ablation_results.csv"),
    index=False
)

position_df.round(4)

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
ax.bar(position_df["Position"], position_df["Accuracy"])
ax.set_xlabel("Attention Position")
ax.set_ylabel("Test Accuracy")
ax.set_ylim(0, 1)
ax.set_title("Channel–Spatial Attention: Position Ablation")
fig.tight_layout()
fig.savefig(
    os.path.join(FIGURES_DIR, "position_ablation_accuracy.pdf"),
    format="pdf",
    dpi=600,
    bbox_inches="tight"
)
plt.show()
plt.close(fig)

In [ ]:
def build_multi_position_model(position="P3_P4", kernel_size=7):
    backbone = ResNet50(
        weights="imagenet",
        include_top=False,
        input_shape=(IMG_SIZE, IMG_SIZE, 3)
    )

    p3_tensor = backbone.get_layer("conv4_block6_out").output
    p4_tensor = backbone.output

    stem = models.Model(backbone.input, p3_tensor)
    tail = models.Model(p3_tensor, p4_tensor)

    inputs = layers.Input(shape=(IMG_SIZE, IMG_SIZE, 3))

    if position == "P4":
        x = models.Model(backbone.input, p4_tensor)(inputs)
        x = channel_attention(x)
        x = spatial_attention(x, kernel_size)

    else:
        x = stem(inputs)

        if position in ["P3", "P3_P4"]:
            x = channel_attention(x)
            x = spatial_attention(x, kernel_size)

        x = tail(x)

        if position == "P3_P4":
            x = channel_attention(x)
            x = spatial_attention(x, kernel_size)

    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dropout(0.3)(x)
    outputs = layers.Dense(NUM_CLASSES, activation="softmax")(x)

    model = models.Model(
        inputs,
        outputs,
        name=f"ResNet50_{position}_Attention"
    )

    return model, stem, tail

In [ ]:
def train_multi_position_model(position, kernel_size=7):
    model, stem, tail = build_multi_position_model(position, kernel_size)

    stem.trainable = False
    tail.trainable = False

    model.compile(
        optimizer=tf.keras.optimizers.Adam(1e-3),
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"]
    )

    frozen_history = model.fit(
        train_ds,
        validation_data=val_ds,
        epochs=EPOCHS_FROZEN,
        callbacks=[
            tf.keras.callbacks.EarlyStopping(
                monitor="val_loss",
                patience=3,
                restore_best_weights=True
            )
        ]
    )

    if position != "P4":
        tail.trainable = True
        for layer in tail.layers:
            layer.trainable = layer.name.startswith("conv5_")
            if isinstance(layer, layers.BatchNormalization):
                layer.trainable = False

    model.compile(
        optimizer=tf.keras.optimizers.Adam(1e-5),
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"]
    )

    fine_history = model.fit(
        train_ds,
        validation_data=val_ds,
        epochs=EPOCHS_FINE,
        callbacks=[
            tf.keras.callbacks.EarlyStopping(
                monitor="val_loss",
                patience=3,
                restore_best_weights=True
            )
        ]
    )

    model.save(
        os.path.join(MODELS_DIR, f"{position}_attention_final.keras")
    )

    with open(
        os.path.join(RESULTS_DIR, f"{position}_attention_history.json"), "w"
    ) as f:
        json.dump({
            "frozen": frozen_history.history,
            "fine_tuning": fine_history.history
        }, f)

    return model, frozen_history, fine_history

In [ ]:
p3_multi_model, p3_multi_history, p3_multi_history_fine = train_multi_position_model("P3")

In [ ]:
p4_multi_model, p4_multi_history, p4_multi_history_fine = train_multi_position_model("P4")

In [ ]:
p3p4_model, p3p4_history, p3p4_history_fine = train_multi_position_model("P3_P4")

In [ ]:
multi_position_models = {
    "P3": channel_spatial_model,
    "P4": p4_model,
    "P3_P4": p3p4_model
}

y_true = np.concatenate([labels.numpy() for _, labels in test_ds])
multi_results = []

for name, model in multi_position_models.items():
    probabilities = model.predict(test_ds, verbose=1)
    y_pred = np.argmax(probabilities, axis=1)

    start = time.perf_counter()
    for images, _ in test_ds:
        model(images, training=False).numpy()
    elapsed = time.perf_counter() - start

    multi_results.append({
        "Model": name,
        "Accuracy": accuracy_score(y_true, y_pred),
        "Macro Precision": precision_score(
            y_true, y_pred, average="macro", zero_division=0
        ),
        "Macro Recall": recall_score(
            y_true, y_pred, average="macro", zero_division=0
        ),
        "Macro F1": f1_score(
            y_true, y_pred, average="macro", zero_division=0
        ),
        "Total Parameters": model.count_params(),
        "Inference Time (ms/image)": elapsed / len(y_true) * 1000
    })

multi_position_df = pd.DataFrame(multi_results)
multi_position_df.to_csv(
    os.path.join(RESULTS_DIR, "multi_position_results.csv"),
    index=False
)

multi_position_df.round(4)

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
ax.bar(multi_position_df["Model"], multi_position_df["Accuracy"])
ax.set_xlabel("Attention Position")
ax.set_ylabel("Test Accuracy")
ax.set_ylim(0, 1)
ax.set_title("Single vs Multiple Attention Positions")
fig.tight_layout()
fig.savefig(
    os.path.join(FIGURES_DIR, "multi_position_accuracy.pdf"),
    format="pdf",
    dpi=600,
    bbox_inches="tight"
)
plt.show()
plt.close(fig)

In [ ]:
spatial_channel_model, history_spatial_channel, history_spatial_channel_fine = train_attention_model(
    "spatial_channel"
)

In [ ]:
order_models = {
    "Channel_Spatial": channel_spatial_model,
    "Spatial_Channel": spatial_channel_model
}

y_true = np.concatenate([labels.numpy() for _, labels in test_ds])
order_results = []

for name, model in order_models.items():
    probabilities = model.predict(test_ds, verbose=1)
    y_pred = np.argmax(probabilities, axis=1)

    start = time.perf_counter()
    for images, _ in test_ds:
        model(images, training=False).numpy()
    elapsed = time.perf_counter() - start

    order_results.append({
        "Attention Order": name,
        "Accuracy": accuracy_score(y_true, y_pred),
        "Macro Precision": precision_score(
            y_true, y_pred, average="macro", zero_division=0
        ),
        "Macro Recall": recall_score(
            y_true, y_pred, average="macro", zero_division=0
        ),
        "Macro F1": f1_score(
            y_true, y_pred, average="macro", zero_division=0
        ),
        "Total Parameters": model.count_params(),
        "Inference Time (ms/image)": elapsed / len(y_true) * 1000
    })

order_df = pd.DataFrame(order_results)
order_df.to_csv(
    os.path.join(RESULTS_DIR, "attention_order_results.csv"),
    index=False
)
order_df.round(4)

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
ax.bar(order_df["Attention Order"], order_df["Accuracy"])
ax.set_ylabel("Test Accuracy")
ax.set_ylim(0, 1)
ax.set_title("Channel–Spatial vs Spatial–Channel Attention")
fig.tight_layout()
fig.savefig(
    os.path.join(FIGURES_DIR, "attention_order_accuracy.pdf"),
    format="pdf",
    dpi=600,
    bbox_inches="tight"
)
plt.show()
plt.close(fig)

In [ ]:
spatial_k3_model, spatial_k3_history, spatial_k3_history_fine = train_attention_model(
    "spatial", kernel_size=3
)

In [ ]:
spatial_k5_model, spatial_k5_history, spatial_k5_history_fine = train_attention_model(
    "spatial", kernel_size=5
)

In [ ]:
spatial_k7_model, spatial_k7_history, spatial_k7_history_fine = train_attention_model(
    "spatial", kernel_size=7
)

In [ ]:
kernel_models = {
    "3x3": spatial_k3_model,
    "5x5": spatial_k5_model,
    "7x7": spatial_k7_model
}

kernel_results = []

for name, model in kernel_models.items():
    probabilities = model.predict(test_ds, verbose=1)
    y_pred = np.argmax(probabilities, axis=1)

    start = time.perf_counter()
    for images, _ in test_ds:
        model(images, training=False).numpy()
    elapsed = time.perf_counter() - start

    kernel_results.append({
        "Spatial Kernel": name,
        "Accuracy": accuracy_score(y_true, y_pred),
        "Macro F1": f1_score(
            y_true, y_pred, average="macro", zero_division=0
        ),
        "Inference Time (ms/image)": elapsed / len(y_true) * 1000
    })

kernel_df = pd.DataFrame(kernel_results)
kernel_df.to_csv(
    os.path.join(RESULTS_DIR, "spatial_kernel_results.csv"),
    index=False
)
kernel_df.round(4)

In [ ]:
fig, ax = plt.subplots(figsize=(7, 5))
ax.bar(kernel_df["Spatial Kernel"], kernel_df["Accuracy"])
ax.set_xlabel("Spatial Attention Kernel")
ax.set_ylabel("Test Accuracy")
ax.set_ylim(0, 1)
ax.set_title("Spatial Attention Kernel-Size Ablation")
fig.tight_layout()
fig.savefig(
    os.path.join(FIGURES_DIR, "spatial_kernel_accuracy.pdf"),
    format="pdf",
    dpi=600,
    bbox_inches="tight"
)
plt.show()
plt.close(fig)

In [ ]:
def se_attention(x, reduction=16):
    channels = int(x.shape[-1])

    weights = layers.GlobalAveragePooling2D()(x)
    weights = layers.Dense(
        max(channels // reduction, 1),
        activation="relu"
    )(weights)
    weights = layers.Dense(channels, activation="sigmoid")(weights)
    weights = layers.Reshape((1, 1, channels))(weights)

    return layers.Multiply()([x, weights])


def eca_attention(x, kernel_size=3):
    channels = int(x.shape[-1])

    weights = layers.GlobalAveragePooling2D()(x)
    weights = layers.Reshape((channels, 1))(weights)
    weights = layers.Conv1D(
        filters=1,
        kernel_size=kernel_size,
        padding="same",
        use_bias=False
    )(weights)
    weights = layers.Activation("sigmoid")(weights)
    weights = layers.Reshape((1, 1, channels))(weights)

    return layers.Multiply()([x, weights])


@tf.keras.utils.register_keras_serializable()
class SpatialPoolCBAM(layers.Layer):
    def call(self, x):
        avg = tf.reduce_mean(x, axis=-1, keepdims=True)
        maximum = tf.reduce_max(x, axis=-1, keepdims=True)
        return tf.concat([avg, maximum], axis=-1)


def cbam_attention(x, reduction=16, kernel_size=7):
    channels = int(x.shape[-1])

    avg = layers.GlobalAveragePooling2D()(x)
    maximum = layers.GlobalMaxPooling2D()(x)

    dense1 = layers.Dense(
        max(channels // reduction, 1),
        activation="relu"
    )
    dense2 = layers.Dense(channels)

    avg_out = dense2(dense1(avg))
    max_out = dense2(dense1(maximum))

    channel_weights = layers.Activation("sigmoid")(
        layers.Add()([avg_out, max_out])
    )
    channel_weights = layers.Reshape((1, 1, channels))(channel_weights)
    x = layers.Multiply()([x, channel_weights])

    spatial_features = SpatialPoolCBAM()(x)
    spatial_weights = layers.Conv2D(
        1,
        kernel_size,
        padding="same",
        activation="sigmoid"
    )(spatial_features)

    return layers.Multiply()([x, spatial_weights])

In [ ]:
def build_section19_model(attention_type):
    backbone = ResNet50(
        weights="imagenet",
        include_top=False,
        input_shape=(IMG_SIZE, IMG_SIZE, 3)
    )

    p3_tensor = backbone.get_layer("conv4_block6_out").output

    stem = models.Model(backbone.input, p3_tensor)
    tail = models.Model(p3_tensor, backbone.output)

    inputs = layers.Input(shape=(IMG_SIZE, IMG_SIZE, 3))
    x = stem(inputs)

    if attention_type == "SE":
        x = se_attention(x)
    elif attention_type == "ECA":
        x = eca_attention(x)
    elif attention_type == "CBAM":
        x = cbam_attention(x)
    else:
        raise ValueError("Supported mechanisms: SE, ECA, CBAM")

    x = tail(x)
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dropout(0.3)(x)
    outputs = layers.Dense(NUM_CLASSES, activation="softmax")(x)

    model = models.Model(
        inputs,
        outputs,
        name=f"ResNet50_P3_{attention_type}"
    )

    return model, stem, tail

In [ ]:
def train_section19_model(attention_type):
    model, stem, tail = build_section19_model(attention_type)

    stem.trainable = False
    tail.trainable = False

    model.compile(
        optimizer=tf.keras.optimizers.Adam(1e-3),
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"]
    )

    frozen_history = model.fit(
        train_ds,
        validation_data=val_ds,
        epochs=EPOCHS_FROZEN,
        callbacks=[
            tf.keras.callbacks.EarlyStopping(
                monitor="val_loss",
                patience=3,
                restore_best_weights=True
            )
        ]
    )

    tail.trainable = True

    for layer in tail.layers:
        layer.trainable = layer.name.startswith("conv5_")
        if isinstance(layer, layers.BatchNormalization):
            layer.trainable = False

    model.compile(
        optimizer=tf.keras.optimizers.Adam(1e-5),
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"]
    )

    fine_history = model.fit(
        train_ds,
        validation_data=val_ds,
        epochs=EPOCHS_FINE,
        callbacks=[
            tf.keras.callbacks.EarlyStopping(
                monitor="val_loss",
                patience=3,
                restore_best_weights=True
            )
        ]
    )

    model.save(
        os.path.join(
            MODELS_DIR,
            f"p3_{attention_type.lower()}_final.keras"
        )
    )

    with open(
        os.path.join(
            RESULTS_DIR,
            f"p3_{attention_type.lower()}_history.json"
        ),
        "w"
    ) as f:
        json.dump({
            "frozen": frozen_history.history,
            "fine_tuning": fine_history.history
        }, f)

    return model, frozen_history, fine_history

In [ ]:
se_model, se_history, se_history_fine = train_section19_model("SE")

In [ ]:
eca_model, eca_history, eca_history_fine = train_section19_model("ECA")

In [ ]:
cbam_model, cbam_history, cbam_history_fine = train_section19_model("CBAM")

In [ ]:
section19_models = {
    "No Attention": baseline_model,
    "Channel Attention": channel_model,
    "Spatial Attention": spatial_model,
    "Channel + Spatial": channel_spatial_model,
    "Self-Attention": self_attention_model,
    "SE": se_model,
    "ECA": eca_model,
    "CBAM": cbam_model
}

y_true = np.concatenate([
    labels.numpy() for _, labels in test_ds
])

section19_results = []

for name, model in section19_models.items():
    probabilities = model.predict(test_ds, verbose=1)
    y_pred = np.argmax(probabilities, axis=1)

    start = time.perf_counter()

    for images, _ in test_ds:
        model(images, training=False).numpy()

    elapsed = time.perf_counter() - start

    section19_results.append({
        "Attention": name,
        "Accuracy": accuracy_score(y_true, y_pred),
        "Precision": precision_score(
            y_true, y_pred, average="macro", zero_division=0
        ),
        "Recall": recall_score(
            y_true, y_pred, average="macro", zero_division=0
        ),
        "Macro F1": f1_score(
            y_true, y_pred, average="macro", zero_division=0
        ),
        "Total Parameters": model.count_params(),
        "Trainable Parameters": sum(
            int(tf.size(v)) for v in model.trainable_variables
        ),
        "Inference Time (ms/image)": (
            elapsed / len(y_true) * 1000
        )
    })

section19_df = pd.DataFrame(section19_results)

section19_df.to_csv(
    os.path.join(RESULTS_DIR, "section19_attention_comparison.csv"),
    index=False
)

section19_df.round(4)

In [ ]:
for metric, title, filename in [
    ("Accuracy", "Test Accuracy", "section19_accuracy.pdf"),
    ("Macro F1", "Macro F1-Score", "section19_macro_f1.pdf")
]:
    fig, ax = plt.subplots(figsize=(12, 6))

    ax.bar(section19_df["Attention"], section19_df[metric])
    ax.set_ylabel(metric)
    ax.set_ylim(0, 1)
    ax.set_title(title)
    ax.tick_params(axis="x", rotation=35)

    fig.tight_layout()
    fig.savefig(
        os.path.join(FIGURES_DIR, filename),
        format="pdf",
        dpi=600,
        bbox_inches="tight"
    )
    plt.show()
    plt.close(fig)

In [ ]:
fig, ax = plt.subplots(figsize=(9, 6))

ax.scatter(
    section19_df["Total Parameters"],
    section19_df["Accuracy"]
)

for _, row in section19_df.iterrows():
    ax.annotate(
        row["Attention"],
        (row["Total Parameters"], row["Accuracy"]),
        fontsize=8
    )

ax.set_xlabel("Total Model Parameters")
ax.set_ylabel("Test Accuracy")
ax.set_title("Accuracy–Complexity Trade-off")

fig.tight_layout()
fig.savefig(
    os.path.join(FIGURES_DIR, "section19_accuracy_vs_parameters.pdf"),
    format="pdf",
    dpi=600,
    bbox_inches="tight"
)
plt.show()
plt.close(fig)

In [ ]:
fig, ax = plt.subplots(figsize=(9, 6))

ax.scatter(
    section19_df["Inference Time (ms/image)"],
    section19_df["Macro F1"]
)

for _, row in section19_df.iterrows():
    ax.annotate(
        row["Attention"],
        (
            row["Inference Time (ms/image)"],
            row["Macro F1"]
        ),
        fontsize=8
    )

ax.set_xlabel("Inference Time (ms/image)")
ax.set_ylabel("Macro F1")
ax.set_title("Macro F1 versus Inference Time")

fig.tight_layout()
fig.savefig(
    os.path.join(FIGURES_DIR, "section19_f1_vs_inference_time.pdf"),
    format="pdf",
    dpi=600,
    bbox_inches="tight"
)
plt.show()
plt.close(fig)

In [ ]:
print("Section 19 comparison:")
print(section19_df.round(4).to_string(index=False))

print("\nSaved files:")
for filename in [
    "section19_attention_comparison.csv",
    "p3_se_final.keras",
    "p3_eca_final.keras",
    "p3_cbam_final.keras"
]:
    folder = RESULTS_DIR if filename.endswith(".csv") else MODELS_DIR
    path = os.path.join(folder, filename)
    print(f"{filename}: {'Saved' if os.path.exists(path) else 'Missing'}")

In [ ]:
section23_models = {
    "Baseline": (baseline_model, "—"),
    "Channel Attention": (channel_model, "P3"),
    "Spatial Attention": (spatial_model, "P3"),
    "Channel+Spatial": (p2_model, "P2"),
    "Channel+Spatial ": (channel_spatial_model, "P3"),
    "Channel+Spatial  ": (p4_model, "P4"),
    "Self-Attention": (self_attention_model, "P3"),
    "Additional Attention-1 (SE)": (se_model, "P3"),
    "Additional Attention-2 (ECA)": (eca_model, "P3")
}

In [ ]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
import time
import numpy as np
import pandas as pd
import os

y_true = np.concatenate([labels.numpy() for _, labels in test_ds])
main_results = []

for name, (model, position) in section23_models.items():
    probabilities = model.predict(test_ds, verbose=0)
    y_pred = np.argmax(probabilities, axis=1)

    for _ in range(2):
        for images, _ in test_ds:
            model(images, training=False)

    start = time.perf_counter()
    total_images = 0

    for images, _ in test_ds:
        model(images, training=False).numpy()
        total_images += int(images.shape[0])

    elapsed = time.perf_counter() - start

    main_results.append({
        "Model": name.strip(),
        "Position": position,
        "Accuracy": accuracy_score(y_true, y_pred),
        "Precision": precision_score(
            y_true, y_pred, average="macro", zero_division=0
        ),
        "Recall": recall_score(
            y_true, y_pred, average="macro", zero_division=0
        ),
        "F1": f1_score(
            y_true, y_pred, average="macro", zero_division=0
        ),
        "Parameters": model.count_params(),
        "Inference Time (ms/image)": elapsed / total_images * 1000
    })

main_results_df = pd.DataFrame(main_results)

main_results_df.to_csv(
    os.path.join(RESULTS_DIR, "section23_main_results.csv"),
    index=False
)

main_results_df.round(4)

In [ ]:
history_groups = {
    "Baseline": (history_frozen, history_fine),
    "Channel": (history_channel, history_channel_fine),
    "Spatial": (history_spatial, history_spatial_fine),
    "Channel+Spatial P3": (
        history_channel_spatial,
        history_channel_spatial_fine
    ),
    "Self-Attention": (history_self, history_self_fine),
    "SE": (se_history, se_history_fine),
    "ECA": (eca_history, eca_history_fine),
    "CBAM": (cbam_history, cbam_history_fine)
}

In [ ]:
def combine_history(history_pair, metric):
    frozen, fine = history_pair

    train_values = frozen.history.get(metric, [])
    fine_values = fine.history.get(metric, [])

    epochs = list(range(1, len(train_values) + 1))
    fine_epochs = list(
        range(len(train_values) + 1,
              len(train_values) + len(fine_values) + 1)
    )

    return epochs + fine_epochs, train_values + fine_values


fig, ax = plt.subplots(figsize=(13, 7))

for name, histories in history_groups.items():
    epochs, train_acc = combine_history(histories, "accuracy")
    _, val_acc = combine_history(histories, "val_accuracy")

    ax.plot(epochs, train_acc, label=f"{name} Train")
    ax.plot(epochs, val_acc, linestyle="--", label=f"{name} Validation")

ax.set_xlabel("Epoch")
ax.set_ylabel("Accuracy")
ax.set_title("Training and Validation Accuracy")
ax.legend(fontsize=8, ncol=2)
ax.grid(True, alpha=0.3)

fig.tight_layout()
fig.savefig(
    os.path.join(FIGURES_DIR, "section24_training_validation_accuracy.pdf"),
    format="pdf",
    dpi=600,
    bbox_inches="tight"
)
plt.show()
plt.close(fig)

In [ ]:
fig, ax = plt.subplots(figsize=(13, 7))

for name, histories in history_groups.items():
    epochs, train_loss = combine_history(histories, "loss")
    _, val_loss = combine_history(histories, "val_loss")

    ax.plot(epochs, train_loss, label=f"{name} Train")
    ax.plot(epochs, val_loss, linestyle="--", label=f"{name} Validation")

ax.set_xlabel("Epoch")
ax.set_ylabel("Categorical Cross-Entropy Loss")
ax.set_title("Training and Validation Loss")
ax.legend(fontsize=8, ncol=2)
ax.grid(True, alpha=0.3)

fig.tight_layout()
fig.savefig(
    os.path.join(FIGURES_DIR, "section24_training_validation_loss.pdf"),
    format="pdf",
    dpi=600,
    bbox_inches="tight"
)
plt.show()
plt.close(fig)

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf

from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

os.makedirs(FIGURES_DIR, exist_ok=True)
os.makedirs(RESULTS_DIR, exist_ok=True)

if "info" in globals() and hasattr(info, "features"):
    class_names = info.features["label"].names
else:
    class_names = ["daisy", "dandelion", "roses", "sunflowers", "tulips"]


def display_image(images):
    image = images.numpy() if tf.is_tensor(images) else np.asarray(images)
    image = image.astype(np.float32)

    # Reverse ResNet50 caffe preprocessing if the image is preprocessed.
    image = image[..., ::-1] + np.array(
        [103.939, 116.779, 123.68], dtype=np.float32
    )
    return np.clip(image, 0, 255).astype(np.uint8)


def save_figure(fig, filename):
    fig.tight_layout()
    fig.savefig(
        os.path.join(FIGURES_DIR, filename),
        format="pdf",
        dpi=600,
        bbox_inches="tight"
    )
    plt.show()
    plt.close(fig)

In [ ]:
selected_models = {
    "Baseline": baseline_model,
    "Channel+Spatial P3": channel_spatial_model
}

y_true = np.concatenate([
    labels.numpy() for _, labels in test_ds
])

confusion_results = {}

for name, model in selected_models.items():
    probabilities = model.predict(test_ds, verbose=1)
    predictions = np.argmax(probabilities, axis=1)

    cm = confusion_matrix(
        y_true,
        predictions,
        labels=np.arange(len(class_names))
    )

    confusion_results[name] = {
        "predictions": predictions,
        "matrix": cm,
        "accuracy": np.mean(predictions == y_true)
    }

    pd.DataFrame(
        cm,
        index=class_names,
        columns=class_names
    ).to_csv(
        os.path.join(
            RESULTS_DIR,
            f"section25_{name.lower().replace('+', '_').replace(' ', '_')}_confusion_matrix.csv"
        )
    )

    fig, ax = plt.subplots(figsize=(8, 7))
    display = ConfusionMatrixDisplay(
        confusion_matrix=cm,
        display_labels=class_names
    )
    display.plot(ax=ax, cmap="Blues", values_format="d", xticks_rotation=45)
    ax.set_title(f"{name}: Confusion Matrix")
    save_figure(
        fig,
        f"section25_{name.lower().replace('+', '_').replace(' ', '_')}_confusion_matrix.pdf"
    )

In [ ]:
from sklearn.metrics import classification_report

class_comparison = []

for name, result in confusion_results.items():
    report = classification_report(
        y_true,
        result["predictions"],
        labels=np.arange(len(class_names)),
        target_names=class_names,
        output_dict=True,
        zero_division=0
    )

    for class_name in class_names:
        class_comparison.append({
            "Model": name,
            "Class": class_name,
            "Precision": report[class_name]["precision"],
            "Recall": report[class_name]["recall"],
            "F1": report[class_name]["f1-score"],
            "Support": report[class_name]["support"]
        })

class_comparison_df = pd.DataFrame(class_comparison)
class_comparison_df.to_csv(
    os.path.join(RESULTS_DIR, "section25_per_class_metrics.csv"),
    index=False
)

class_comparison_df.round(4)

In [ ]:
# =====================================================================
# SECTION 27 - Grad-CAM Interpretation
# Paste these cells AFTER Section 25/26 and BEFORE Section 28
# (the cell that starts with `def find_channel_weight_layer`).
# Needs: baseline_model, channel_spatial_model, test_idx,
#        test_ds, class_names, FIGURES_DIR, RESULTS_DIR, IMG_SIZE, SEED
# =====================================================================

# %% [markdown]
# ## 27. Grad-CAM Interpretation

import numpy as np
import tensorflow as tf

# %% Cell 1 - Grad-CAM engine (self-contained: does not rely on the globals `images` / `labels`)
import numpy as np
import pandas as pd
import tensorflow as tf
import tensorflow_datasets as tfds
import matplotlib.pyplot as plt
from tensorflow.keras import layers
from tensorflow.keras.applications.resnet50 import preprocess_input

# Reload ONLY the test images (uint8, original order, same resize as the notebook), so earlier
# cells that overwrite `images` (e.g. `for images, _ in test_ds`) can't break this section.
_test_pos = {int(i): k for k, i in enumerate(np.asarray(test_idx))}      # dataset index -> test position
gc_test_images = np.zeros((len(_test_pos), IMG_SIZE, IMG_SIZE, 3), dtype=np.uint8)
gc_test_labels = np.zeros(len(_test_pos), dtype=np.int64)
_raw = tfds.load("tf_flowers", split="train", as_supervised=True)
for _i, (_im, _lb) in enumerate(tfds.as_numpy(_raw)):
    if _i in _test_pos:
        _k = _test_pos[_i]
        gc_test_images[_k] = tf.image.resize(_im, (IMG_SIZE, IMG_SIZE)).numpy().astype(np.uint8)
        gc_test_labels[_k] = _lb
del _raw
print("Grad-CAM test images:", gc_test_images.shape)

# Models to compare. First = baseline, second = attention model of interest.
# You can add SE / ECA / CBAM / P2 / P4 etc. here for extra comparisons.
GRADCAM_MODELS = {
    "Baseline": baseline_model,
    "Channel+Spatial P3": channel_spatial_model,
}
BASE_NAME, ATT_NAME = list(GRADCAM_MODELS.keys())[:2]


def build_gradcam_model(model):
    """
    Grad-CAM target = feature map that enters the final GlobalAveragePooling2D
    (7x7x2048, i.e. the last ResNet conv5 block, *after* any attention that
    sits there). Because stem/tail/ResNet50 are nested Models in this notebook,
    we use the tensor in the OUTER graph instead of an inner layer name.
    """
    gap_layers = [l for l in model.layers
                  if isinstance(l, layers.GlobalAveragePooling2D)]
    if not gap_layers:
        raise ValueError("No GlobalAveragePooling2D layer found in model.")
    feature_tensor = gap_layers[-1].input          # final GAP = classifier GAP
    return tf.keras.Model(model.inputs, [feature_tensor, model.output])


def make_gradcam(grad_model, x_batch, class_index=None):
    """x_batch: preprocessed (1, H, W, 3). Returns cam[H,W] in [0,1], probs, class."""
    x = tf.convert_to_tensor(x_batch, dtype=tf.float32)
    with tf.GradientTape() as tape:
        tape.watch(x)                               # works even if all layers are frozen
        feature_maps, preds = grad_model(x, training=False)
        if class_index is None:
            class_index = int(tf.argmax(preds[0]))
        class_score = preds[:, class_index]

    grads = tape.gradient(class_score, feature_maps)
    weights = tf.reduce_mean(grads, axis=(1, 2), keepdims=True)   # GAP of gradients
    cam = tf.reduce_sum(weights * feature_maps, axis=-1)          # weighted sum
    cam = tf.nn.relu(cam)[0]
    cam = cam / (tf.reduce_max(cam) + 1e-8)
    cam = tf.image.resize(cam[..., tf.newaxis], (IMG_SIZE, IMG_SIZE),
                          method="bilinear")[..., 0]
    return cam.numpy(), preds[0].numpy(), class_index


def overlay_cam(image_uint8, cam, alpha=0.40):
    heat = plt.get_cmap("jet")(cam)[..., :3]
    img = image_uint8.astype(np.float32) / 255.0
    return np.clip((1 - alpha) * img + alpha * heat, 0, 1)


def prep(image_uint8):
    return preprocess_input(image_uint8[np.newaxis].astype(np.float32))


grad_models = {name: build_gradcam_model(m) for name, m in GRADCAM_MODELS.items()}

# quick sanity check
_cam, _p, _c = make_gradcam(grad_models[BASE_NAME], prep(gc_test_images[0]))
print("Grad-CAM OK:", _cam.shape, "| predicted:", class_names[_c])


# %% Cell 2 - Predictions on the test set + choose images to interpret
y_test = gc_test_labels
pred_cls = {}
for name, m in GRADCAM_MODELS.items():
    probs = m.predict(test_ds, verbose=0)           # test_ds is unshuffled -> same order as test_idx
    pred_cls[name] = np.argmax(probs, axis=1)

pb, pa = pred_cls[BASE_NAME], pred_cls[ATT_NAME]
rng = np.random.default_rng(SEED)

# (a) one image per class that BOTH models classify correctly
correct_samples = []
for c in range(len(class_names)):
    pool = np.where((y_test == c) & (pb == c) & (pa == c))[0]
    if len(pool):
        correct_samples.append(int(rng.choice(pool)))

# (b) cases where the two models disagree / both fail
def pick(mask, k=3):
    pool = np.where(mask)[0]
    return [int(i) for i in rng.choice(pool, size=min(k, len(pool)), replace=False)] if len(pool) else []

attention_fixes   = pick((pb != y_test) & (pa == y_test))   # baseline wrong, attention right
attention_hurts   = pick((pb == y_test) & (pa != y_test))   # baseline right, attention wrong
both_wrong        = pick((pb != y_test) & (pa != y_test))

print("Correct (both):", len(correct_samples), "| Attention fixes:", len(attention_fixes),
      "| Attention hurts:", len(attention_hurts), "| Both wrong:", len(both_wrong))

# Samples used by Section 28 (they were used but never defined earlier)
original_image = gc_test_images[correct_samples[0]]
sample_image = prep(original_image)


# %% Cell 3 - Grad-CAM comparison figures
def plot_gradcam_grid(sample_positions, title, filename, target="pred"):
    """target='pred' -> explain each model's own prediction; 'true' -> explain the true class."""
    n = len(sample_positions)
    if n == 0:
        print("Nothing to plot for:", title)
        return
    fig, axes = plt.subplots(n, 3, figsize=(11, 3.5 * n), squeeze=False)

    for r, pos in enumerate(sample_positions):
        img = gc_test_images[pos]
        true_c = int(y_test[pos])
        x = prep(img)

        axes[r, 0].imshow(img)
        axes[r, 0].set_title(f"Original\nTrue: {class_names[true_c]}", fontsize=10)

        for c_idx, name in enumerate([BASE_NAME, ATT_NAME], start=1):
            cls = true_c if target == "true" else None
            cam, probs, used_cls = make_gradcam(grad_models[name], x, cls)
            pred_c = int(np.argmax(probs))
            colour = "green" if pred_c == true_c else "red"
            axes[r, c_idx].imshow(overlay_cam(img, cam))
            axes[r, c_idx].set_title(
                f"{name}\nPred: {class_names[pred_c]} ({probs[pred_c]:.2f})",
                fontsize=10, color=colour
            )
        for ax in axes[r]:
            ax.axis("off")

    fig.suptitle(title, fontsize=13, y=1.0)
    fig.tight_layout()
    fig.savefig(f"{FIGURES_DIR}/{filename}", format="pdf", dpi=600, bbox_inches="tight")
    plt.show()
    plt.close(fig)


plot_gradcam_grid(correct_samples,
                  "Grad-CAM: images correctly classified by both models",
                  "section27_gradcam_correct.pdf")

plot_gradcam_grid(attention_fixes + attention_hurts + both_wrong,
                  "Grad-CAM: baseline vs attention disagreements / failures",
                  "section27_gradcam_disagreements.pdf")


# %% Cell 4 - Grad-CAM for EVERY model on one image (optional)
def plot_all_models_one_image(pos, filename):
    img = gc_test_images[pos]
    x = prep(img)
    names = list(GRADCAM_MODELS.keys())
    fig, axes = plt.subplots(1, len(names) + 1, figsize=(4 * (len(names) + 1), 4))
    axes[0].imshow(img)
    axes[0].set_title(f"Original\nTrue: {class_names[int(y_test[pos])]}")
    for ax, name in zip(axes[1:], names):
        cam, probs, _ = make_gradcam(grad_models[name], x)
        p = int(np.argmax(probs))
        ax.imshow(overlay_cam(img, cam))
        ax.set_title(f"{name}\nPred: {class_names[p]} ({probs[p]:.2f})", fontsize=10)
    for ax in axes:
        ax.axis("off")
    fig.tight_layout()
    fig.savefig(f"{FIGURES_DIR}/{filename}", format="pdf", dpi=600, bbox_inches="tight")
    plt.show()
    plt.close(fig)

plot_all_models_one_image(correct_samples[0], "section27_gradcam_all_models.pdf")


# %% Cell 5 - Optional numbers to SUPPORT (not replace) your visual observations
# center_mass : share of Grad-CAM energy inside the central 50%x50% box
#               (a uniform map would give 0.25; higher => more central focus)
# top10_mass  : share of energy in the hottest 10% of pixels
#               (higher => concentrated; lower => spread over the whole image)
def cam_stats(cam):
    total = cam.sum() + 1e-8
    lo, hi = IMG_SIZE // 4, 3 * IMG_SIZE // 4
    center_mass = cam[lo:hi, lo:hi].sum() / total
    k = int(0.10 * cam.size)
    top10_mass = np.sort(cam.ravel())[::-1][:k].sum() / total
    return center_mass, top10_mass

rows = []
for name in [BASE_NAME, ATT_NAME]:
    cms, t10s = [], []
    for pos in range(len(test_idx)):
        cam, _, _ = make_gradcam(grad_models[name], prep(gc_test_images[pos]))
        cm_, t10 = cam_stats(cam)
        cms.append(cm_); t10s.append(t10)
    rows.append({"Model": name,
                 "Mean center mass": np.mean(cms),
                 "Mean top-10% mass": np.mean(t10s)})

gradcam_stats_df = pd.DataFrame(rows)
gradcam_stats_df.to_csv(f"{RESULTS_DIR}/section27_gradcam_stats.csv", index=False)
gradcam_stats_df.round(4)


# %% [markdown]
# ### 27.1 Observations (write ONLY what you can see in the figures)
#
# Template - replace the bracketed parts after looking at the saved figures:
#
# - **Correctly classified images:** The baseline network activates [flower + surrounding leaves / the whole image],
#   whereas the Channel+Spatial P3 model produces [stronger / weaker] activation over the [petal region /
#   central flower structure] and [weaker / similar] activation over the [background / leaves].
# - **Where the models disagree:** For [image/class], the baseline predicted [class] with activation on [leaves /
#   background objects], while the attention model predicted [class] with activation on [petals / flower centre].
# - **Failure cases:** When the attention model was wrong, its activation [shifted to ... / spread across ...].
#
# Avoid: "The Grad-CAM of Model A is better."  Prefer: describe the visible region, strength and spread.

In [ ]:
def find_channel_weight_layer(model):
    candidates = []

    for layer in model.layers:
        if isinstance(layer, layers.Dense):
            try:
                shape = tuple(layer.output.shape)
                if (
                    len(shape) == 4
                    and shape[-1] is not None
                    and layer.activation == tf.keras.activations.sigmoid
                ):
                    candidates.append(layer)
            except (AttributeError, ValueError):
                pass

    if not candidates:
        raise ValueError(
            "Could not locate the channel-weight layer. "
            "Inspect model.layers and select the sigmoid Dense layer "
            "inside the channel-attention module."
        )

    return candidates[-1]


channel_weight_layer = find_channel_weight_layer(channel_model)

channel_weight_model = tf.keras.Model(
    inputs=channel_model.inputs,
    outputs=channel_weight_layer.output
)

channel_weights = channel_weight_model.predict(sample_image, verbose=0)
channel_weights = np.asarray(channel_weights).reshape(-1)

top_k = min(20, len(channel_weights))
top_indices = np.argsort(channel_weights)[-top_k:][::-1]

fig, ax = plt.subplots(figsize=(10, 5))
ax.bar(
    [str(i) for i in top_indices],
    channel_weights[top_indices]
)
ax.set_xlabel("Channel Index")
ax.set_ylabel("Attention Weight")
ax.set_title(f"Top {top_k} Channel-Attention Weights")
save_figure(fig, "section28_channel_attention_top20.pdf")

pd.DataFrame({
    "Channel Index": top_indices,
    "Attention Weight": channel_weights[top_indices]
}).to_csv(
    os.path.join(RESULTS_DIR, "section28_channel_attention_weights.csv"),
    index=False
)

In [ ]:
def find_spatial_weight_layer(model):
    candidates = []

    for layer in model.layers:
        if isinstance(layer, layers.Conv2D):
            try:
                shape = tuple(layer.output.shape)
                if (
                    len(shape) == 4
                    and shape[-1] == 1
                    and layer.activation == tf.keras.activations.sigmoid
                ):
                    candidates.append(layer)
            except (AttributeError, ValueError):
                pass

    if not candidates:
        raise ValueError(
            "Could not locate the spatial-attention Conv2D layer. "
            "Inspect model.layers and select the sigmoid 1-channel convolution."
        )

    return candidates[-1]


spatial_weight_layer = find_spatial_weight_layer(spatial_model)

spatial_weight_model = tf.keras.Model(
    inputs=spatial_model.inputs,
    outputs=spatial_weight_layer.output
)

spatial_map = spatial_weight_model.predict(sample_image, verbose=0)[0, :, :, 0]
spatial_map = tf.image.resize(
    spatial_map[..., np.newaxis],
    (IMG_SIZE, IMG_SIZE)
).numpy()[..., 0]

fig, axes = plt.subplots(1, 3, figsize=(15, 5))

axes[0].imshow(original_image)
axes[0].set_title("Input Image")

axes[1].imshow(spatial_map, cmap="jet")
axes[1].set_title("Spatial Attention Map")

axes[2].imshow(original_image)
axes[2].imshow(spatial_map, cmap="jet", alpha=0.45)
axes[2].set_title("Spatial Attention Overlay")

for ax in axes:
    ax.axis("off")

save_figure(fig, "section28_spatial_attention_visualization.pdf")

In [ ]:
self_attention_layers = [
    layer for layer in self_attention_model.layers
    if isinstance(layer, LightweightSelfAttention)
]

if not self_attention_layers:
    raise ValueError(
        "No LightweightSelfAttention layer was found in self_attention_model."
    )

self_layer = self_attention_layers[0]

self_response_model = tf.keras.Model(
    inputs=self_attention_model.inputs,
    outputs=[self_layer.input, self_layer.output]
)

features_before, features_after = self_response_model.predict(
    sample_image,
    verbose=0
)

response_map = np.mean(
    np.abs(features_after[0] - features_before[0]),
    axis=-1
)

response_map = tf.image.resize(
    response_map[..., np.newaxis],
    (IMG_SIZE, IMG_SIZE)
).numpy()[..., 0]

fig, axes = plt.subplots(1, 3, figsize=(15, 5))

axes[0].imshow(original_image)
axes[0].set_title("Input Image")

axes[1].imshow(response_map, cmap="viridis")
axes[1].set_title("Aggregated Self-Attention Response")

axes[2].imshow(original_image)
axes[2].imshow(response_map, cmap="jet", alpha=0.45)
axes[2].set_title("Response Overlay")

for ax in axes:
    ax.axis("off")

save_figure(fig, "section28_self_attention_response.pdf")

In [ ]:
position_models = {
    "P2": p2_model,
    "P3": channel_spatial_model,
    "P4": p4_model
}

In [ ]:
def extract_spatial_attention_map(model, image_batch):
    candidates = []

    for layer in model.layers:
        if isinstance(layer, tf.keras.layers.Conv2D):
            if layer.filters == 1 and layer.activation == tf.keras.activations.sigmoid:
                candidates.append(layer)

    if not candidates:
        raise ValueError(
            f"No single-channel sigmoid Conv2D layer found in {model.name}. "
            "Inspect the model layers to identify the spatial-attention output."
        )

    attention_layer = candidates[-1]

    attention_model = tf.keras.Model(
        inputs=model.inputs,
        outputs=attention_layer.output
    )

    attention_map = attention_model.predict(image_batch, verbose=0)[0, :, :, 0]

    attention_map = tf.image.resize(
        attention_map[..., np.newaxis],
        (IMG_SIZE, IMG_SIZE)
    ).numpy()[..., 0]

    return attention_map, attention_layer.name


sample_image = None
sample_label = None

for images, labels in test_ds.take(1):
    sample_image = images[0:1]
    sample_label = int(labels[0].numpy())

original_image = display_image(sample_image[0])

fig, axes = plt.subplots(1, 4, figsize=(18, 5))

axes[0].imshow(original_image)
axes[0].set_title(f"Input: {class_names[sample_label]}")
axes[0].axis("off")

position_attention_maps = {}

for index, (position, model) in enumerate(position_models.items(), start=1):
    attention_map, layer_name = extract_spatial_attention_map(
        model, sample_image
    )

    position_attention_maps[position] = attention_map

    axes[index].imshow(original_image)
    axes[index].imshow(attention_map, cmap="jet", alpha=0.5)
    axes[index].set_title(f"{position} Spatial Attention")
    axes[index].axis("off")

save_figure(fig, "section29_attention_position_visualization.pdf")

for position, attention_map in position_attention_maps.items():
    np.save(
        os.path.join(RESULTS_DIR, f"section29_{position.lower()}_attention.npy"),
        attention_map
    )
    print(position, "map shape:", attention_map.shape)

In [ ]:

# SECTION 30: Qualitative Comparison Across Attention Mechanisms
# 3 correctly classified + 2 incorrectly classified baseline examples
#
# Columns:
# Input | Baseline Grad-CAM | Channel Grad-CAM |
# Spatial Attention | Channel+Spatial Grad-CAM | Self-Attention Response

import os
import numpy as np
import pandas as pd
import tensorflow as tf
import matplotlib.pyplot as plt

os.makedirs(FIGURES_DIR, exist_ok=True)
os.makedirs(RESULTS_DIR, exist_ok=True)

# ---------------------------------------------------------
# 1. Models used in the comparison
# ---------------------------------------------------------
qualitative_models = {
    "Baseline": baseline_model,
    "Channel": channel_model,
    "Spatial": spatial_model,
    "Channel+Spatial P3": channel_spatial_model,
    "Self-Attention": self_attention_model,
}

# ---------------------------------------------------------
# 2. Collect the fixed test set
# ---------------------------------------------------------
test_images_list = []
test_labels_list = []

for images, labels in test_ds:
    test_images_list.append(images.numpy())
    test_labels_list.append(labels.numpy())

test_images = np.concatenate(test_images_list, axis=0)
test_labels = np.concatenate(test_labels_list, axis=0)

# ---------------------------------------------------------
# 3. Select 3 baseline-correct and 2 baseline-incorrect images
# ---------------------------------------------------------
baseline_predictions_all = np.argmax(
    baseline_model.predict(
        test_images, batch_size=BATCH_SIZE, verbose=1
    ),
    axis=1
)

correct_indices = np.where(
    baseline_predictions_all == test_labels
)[0][:3]

incorrect_indices = np.where(
    baseline_predictions_all != test_labels
)[0][:2]

if len(correct_indices) < 3 or len(incorrect_indices) < 2:
    raise ValueError(
        "Could not find at least three baseline-correct and "
        "two baseline-incorrect test images."
    )

selected_indices = list(correct_indices) + list(incorrect_indices)

print("Baseline-correct image indices:", correct_indices.tolist())
print("Baseline-incorrect image indices:", incorrect_indices.tolist())

# ---------------------------------------------------------
# 4. Prepare connected Grad-CAM graphs
# ---------------------------------------------------------
gradcam_graphs = {
    "Baseline": build_gradcam_graph(baseline_model, "baseline"),
    "Channel": build_gradcam_graph(channel_model, "channel"),
    "Spatial": build_gradcam_graph(spatial_model, "spatial"),
    "Channel+Spatial P3": build_gradcam_graph(
        channel_spatial_model, "channel_spatial"
    ),
}

# ---------------------------------------------------------
# 5. Helpers
# ---------------------------------------------------------
def to_rgb(image):
    rgb = display_image(image)
    if tf.is_tensor(rgb):
        rgb = rgb.numpy()

    rgb = np.asarray(rgb)

    if rgb.max() <= 1.0:
        rgb = rgb * 255.0

    return np.clip(rgb, 0, 255).astype(np.uint8)


def resize_map(heatmap, target_size=(224, 224)):
    return tf.image.resize(
        np.asarray(heatmap)[..., np.newaxis],
        target_size,
        method="bilinear"
    ).numpy().squeeze()


def get_spatial_attention_map_for_image(image_batch):
    # Reuse the trained spatial-attention model's layers
    inputs = tf.keras.Input(shape=(224, 224, 3))

    x = spatial_model.get_layer("functional_2")(inputs)
    x = spatial_model.get_layer("spatial_attention_pooling")(x)
    attention = spatial_model.get_layer("conv2d")(x)

    attention_model = tf.keras.Model(inputs, attention)

    result = attention_model(image_batch, training=False)[0, :, :, 0]
    return result.numpy()


def get_self_attention_response(image_batch):
    """
    Return an internal self-attention response when the model exposes
    a layer named 'self_attention' or 'self_attention_layer'.

    The function intentionally raises an error if the actual layer
    cannot be identified; it does not substitute Grad-CAM and label
    it as self-attention.
    """
    candidates = [
        layer for layer in self_attention_model.layers
        if "self_attention" in layer.name.lower()
    ]

    if not candidates:
        raise ValueError(
            "Could not identify the self-attention layer. "
            "Inspect self_attention_model.summary() and set the "
            "correct layer name in get_self_attention_response()."
        )

    attention_layer = candidates[0]

    # Use the actual layer's output if its tensor is connected to the
    # model's outer graph.
    attention_model = tf.keras.Model(
        inputs=self_attention_model.inputs,
        outputs=attention_layer.output
    )

    response = attention_model(image_batch, training=False)

    if isinstance(response, (tuple, list)):
        response = response[-1]

    response = tf.convert_to_tensor(response)

    # Convert feature response into a 2D spatial map.
    if len(response.shape) == 4:
        response = tf.reduce_mean(tf.abs(response[0]), axis=-1)
    elif len(response.shape) == 3:
        response = tf.reduce_mean(tf.abs(response[0]), axis=-1)
    elif len(response.shape) == 2:
        response = tf.reshape(
            tf.abs(response[0]),
            (int(tf.sqrt(tf.cast(tf.shape(response)[-1], tf.float32))),
             int(tf.sqrt(tf.cast(tf.shape(response)[-1], tf.float32))))
        )
    else:
        raise ValueError(
            f"Unsupported self-attention output shape: {response.shape}"
        )

    response = response / (
        tf.reduce_max(response) + tf.keras.backend.epsilon()
    )

    return response.numpy()


# ---------------------------------------------------------
# 6. Generate prediction records for all five images
# ---------------------------------------------------------
qualitative_records = []

for index in selected_indices:
    image_batch = test_images[index:index + 1]
    true_label = int(test_labels[index])

    for model_name, model in qualitative_models.items():
        probabilities = model.predict(image_batch, verbose=0)[0]
        predicted_label = int(np.argmax(probabilities))

        qualitative_records.append({
            "Image Index": int(index),
            "Example Type": (
                "Baseline correct"
                if index in correct_indices
                else "Baseline incorrect"
            ),
            "Model": model_name,
            "True Class": class_names[true_label],
            "Predicted Class": class_names[predicted_label],
            "Confidence": float(probabilities[predicted_label]),
            "Correct": predicted_label == true_label
        })

qualitative_df = pd.DataFrame(qualitative_records)

csv_path = os.path.join(
    RESULTS_DIR,
    "section30_qualitative_comparison.csv"
)
qualitative_df.to_csv(csv_path, index=False)

print("\nPrediction table saved:", csv_path)
from IPython.display import display as show_table
show_table(qualitative_df.round(4))

# ---------------------------------------------------------
# 7. Create the 5-row × 6-column visualization
# ---------------------------------------------------------
fig, axes = plt.subplots(
    nrows=5,
    ncols=6,
    figsize=(25, 19)
)

column_titles = [
    "Input Image",
    "Baseline Grad-CAM",
    "Channel Grad-CAM",
    "Spatial Attention Map",
    "Channel+Spatial Grad-CAM",
    "Self-Attention Response",
]

for col, title in enumerate(column_titles):
    axes[0, col].set_title(title, fontsize=11, fontweight="bold")

for row, index in enumerate(selected_indices):
    image_batch = test_images[index:index + 1]
    true_label = int(test_labels[index])
    rgb = to_rgb(image_batch[0])

    example_type = (
        "Baseline correct"
        if index in correct_indices
        else "Baseline incorrect"
    )

    # Input
    axes[row, 0].imshow(rgb)
    axes[row, 0].set_ylabel(
        f"Image {index}\nTrue: {class_names[true_label]}\n{example_type}",
        fontsize=9
    )

    # Grad-CAM for baseline, channel, and channel+spatial
    for col, model_name in [
        (1, "Baseline"),
        (2, "Channel"),
        (4, "Channel+Spatial P3"),
    ]:
        heatmap, _ = generate_gradcam(
            gradcam_graphs[model_name],
            image_batch
        )
        heatmap = resize_map(heatmap)

        axes[row, col].imshow(rgb)
        axes[row, col].imshow(
            heatmap, cmap="jet", alpha=0.45, vmin=0, vmax=1
        )

    # Actual spatial attention map from the spatial-attention module
    spatial_map = get_spatial_attention_map_for_image(image_batch)
    spatial_map = resize_map(spatial_map)

    axes[row, 3].imshow(rgb)
    axes[row, 3].imshow(
        spatial_map, cmap="jet", alpha=0.45, vmin=0, vmax=1
    )

    # Actual self-attention response
    try:
        self_map = get_self_attention_response(image_batch)
        self_map = resize_map(self_map)

        axes[row, 5].imshow(rgb)
        axes[row, 5].imshow(
            self_map, cmap="jet", alpha=0.45, vmin=0, vmax=1
        )
    except Exception as exc:
        axes[row, 5].text(
            0.5, 0.5,
            f"Self-attention map unavailable\n{str(exc)[:100]}",
            ha="center", va="center", wrap=True,
            transform=axes[row, 5].transAxes, fontsize=8
        )

    # Add predictions and correctness labels to model panels
    for col, model_name in [
        (1, "Baseline"),
        (2, "Channel"),
        (3, "Spatial"),
        (4, "Channel+Spatial P3"),
        (5, "Self-Attention"),
    ]:
        model = qualitative_models[model_name]
        probs = model.predict(image_batch, verbose=0)[0]
        pred = int(np.argmax(probs))
        status = "Correct" if pred == true_label else "Incorrect"

        axes[row, col].set_xlabel(
            f"Pred: {class_names[pred]}\n"
            f"Confidence: {probs[pred]:.2f} | {status}",
            fontsize=8
        )

    for col in range(6):
        axes[row, col].axis("off")

fig.suptitle(
    "Section 30: Qualitative Comparison Across Attention Mechanisms",
    fontsize=16,
    y=1.01
)

fig.tight_layout()

pdf_path = os.path.join(
    FIGURES_DIR,
    "section30_qualitative_model_comparison.pdf"
)

fig.savefig(
    pdf_path,
    format="pdf",
    dpi=600,
    bbox_inches="tight"
)

plt.show()
plt.close(fig)

print("\nSaved visualization:", pdf_path)
print("Saved prediction table:", csv_path)


In [ ]:
for filename in [
    "section29_attention_position_visualization.pdf",
    "section30_qualitative_comparison.csv",
    "section30_qualitative_model_comparison.pdf"
]:
    folder = RESULTS_DIR if filename.endswith(".csv") else FIGURES_DIR
    path = os.path.join(folder, filename)
    print(f"{filename}: {'Saved' if os.path.exists(path) else 'Missing'}")

In [ ]:
def save_feature_map_comparison(
    model,
    image_batch,
    filename="section26_feature_maps_before_after_attention.pdf",
    num_channels=8
):
    backbone = model.get_layer("functional_4")

    p3_extractor = tf.keras.Model(
        inputs=backbone.input,
        outputs=backbone.output
    )

    input_image = image_batch[:1]

    feature_before = p3_extractor(input_image, training=False)

    attention_model = tf.keras.Model(
        inputs=model.input,
        outputs=model.get_layer("multiply_3").output
    )

    feature_after = attention_model(input_image, training=False)

    before = feature_before[0].numpy()
    after = feature_after[0].numpy()

    if before.shape != after.shape:
        raise ValueError(
            f"Feature shapes differ: before={before.shape}, "
            f"after={after.shape}"
        )

    channel_strength = np.mean(np.abs(before), axis=(0, 1))
    selected_channels = np.argsort(channel_strength)[-num_channels:][::-1]

    fig, axes = plt.subplots(
        2, num_channels + 1,
        figsize=(2.2 * (num_channels + 1), 5.5)
    )

    axes[0, 0].imshow(display_image(input_image[0]))
    axes[0, 0].set_title("Input")
    axes[0, 0].axis("off")

    axes[1, 0].imshow(display_image(input_image[0]))
    axes[1, 0].set_title("Input")
    axes[1, 0].axis("off")

    for col, channel in enumerate(selected_channels, start=1):
        before_map = before[:, :, channel]
        after_map = after[:, :, channel]

        axes[0, col].imshow(before_map, cmap="viridis")
        axes[0, col].set_title(f"Before\nCh {channel}")
        axes[0, col].axis("off")

        axes[1, col].imshow(after_map, cmap="viridis")
        axes[1, col].set_title(f"After\nCh {channel}")
        axes[1, col].axis("off")

    axes[0, 0].set_ylabel("Before attention", fontsize=11)
    axes[1, 0].set_ylabel("After attention", fontsize=11)

    fig.suptitle(
        "P3 Feature Maps Before and After Channel+Spatial Attention",
        fontsize=14
    )
    fig.tight_layout()

    save_figure(fig, filename)
    plt.show()
    plt.close(fig)

    print("Feature shape before attention:", before.shape)
    print("Feature shape after attention:", after.shape)
    print("Selected channels:", selected_channels.tolist())


save_feature_map_comparison(
    channel_spatial_model,
    sample_image
)


In [ ]:
for layer in channel_spatial_model.layers:
    print(
        layer.name,
        type(layer).__name__,
        getattr(layer, "output", None).shape
        if getattr(layer, "output", None) is not None
        else None
    )


In [ ]:

def get_spatial_attention_map(model, image_batch):
    pooling_layer = next(
        layer for layer in model.layers
        if isinstance(layer, tf.keras.layers.Layer)
        and "spatial_attention_pooling" in layer.name
    )

    pooling_index = model.layers.index(pooling_layer)

    spatial_conv = None
    for layer in model.layers[pooling_index + 1:]:
        if isinstance(layer, tf.keras.layers.Conv2D) and layer.filters == 1:
            spatial_conv = layer
            break

    if spatial_conv is None:
        raise ValueError("Spatial-attention Conv2D layer was not found.")

    attention_model = tf.keras.Model(
        inputs=model.inputs,
        outputs=spatial_conv.output
    )

    attention_map = attention_model.predict(image_batch, verbose=0)[0, :, :, 0]

    attention_map = tf.image.resize(
        attention_map[..., np.newaxis],
        (IMG_SIZE, IMG_SIZE)
    ).numpy()[..., 0]

    return attention_map



In [ ]:
import tensorflow as tf

def build_gradcam_graph(model, model_type):
    inputs = tf.keras.Input(shape=(224, 224, 3))

    if model_type == "baseline":
        x = model.get_layer("resnet50")(inputs)
        target_features = x
        x = model.get_layer("global_average_pooling2d")(x)
        x = model.get_layer("dropout")(x)
        predictions = model.get_layer("dense")(x)

    elif model_type == "channel":
        x = model.get_layer("functional")(inputs)
        target_features = x

        weights = model.get_layer("global_average_pooling2d")(x)
        weights = model.get_layer("dense")(weights)
        weights = model.get_layer("dense_1")(weights)
        x = model.get_layer("multiply")([x, weights])

        x = model.get_layer("functional_1")(x)
        x = model.get_layer("global_average_pooling2d_1")(x)
        x = model.get_layer("dropout_1")(x) if any(
            layer.name == "dropout_1" for layer in model.layers
        ) else model.get_layer("dropout")(x)
        predictions = model.get_layer("dense_2")(x)

    elif model_type == "spatial":
        x = model.get_layer("functional_2")(inputs)
        target_features = x

        attention = model.get_layer("spatial_attention_pooling")(x)
        attention = model.get_layer("conv2d")(attention)
        x = model.get_layer("multiply_1")([x, attention])

        x = model.get_layer("functional_3")(x)
        x = model.get_layer("global_average_pooling2d_2")(x)
        x = model.get_layer("dropout_1")(x)
        predictions = model.get_layer("dense_3")(x)

    elif model_type == "channel_spatial":
        x = model.get_layer("functional_4")(inputs)
        target_features = x

        weights = model.get_layer("global_average_pooling2d_3")(x)
        weights = model.get_layer("dense_4")(weights)
        weights = model.get_layer("dense_5")(weights)
        x = model.get_layer("multiply_2")([x, weights])

        attention = model.get_layer("spatial_attention_pooling_1")(x)
        attention = model.get_layer("conv2d_1")(attention)
        x = model.get_layer("multiply_3")([x, attention])

        x = model.get_layer("functional_5")(x)
        x = model.get_layer("global_average_pooling2d_4")(x)
        x = model.get_layer("dropout_2")(x)
        predictions = model.get_layer("dense_6")(x)

    else:
        raise ValueError(f"Unknown model type: {model_type}")

    return tf.keras.Model(inputs, [target_features, predictions])

In [ ]:
gradcam_models = {
    "Baseline": (baseline_model, "baseline"),
    "Channel": (channel_model, "channel"),
    "Spatial": (spatial_model, "spatial"),
    "Channel + Spatial": (channel_spatial_model, "channel_spatial"),
}

for name, (model, model_type) in gradcam_models.items():
    try:
        grad_model = build_gradcam_graph(model, model_type)
        features, predictions = grad_model(sample_image, training=False)

        print(
            f"{name}: features={features.shape}, "
            f"predictions={predictions.shape}, "
            f"max probability={float(tf.reduce_max(predictions[0])):.4f}"
        )
    except Exception as e:
        print(f"{name}: {type(e).__name__}: {e}")

In [ ]:
import numpy as np
import tensorflow as tf
import matplotlib.pyplot as plt
import os

def make_gradcam_heatmap(grad_model, image, class_index=None):
    with tf.GradientTape() as tape:
        features, predictions = grad_model(image, training=False)

        if class_index is None:
            class_index = tf.argmax(predictions[0])

        class_score = predictions[:, class_index]

    gradients = tape.gradient(class_score, features)

    if gradients is None:
        raise ValueError("Gradients are None. Check the Grad-CAM graph.")

    # Average gradients across spatial dimensions
    channel_weights = tf.reduce_mean(gradients, axis=(1, 2))

    # Weighted combination of feature maps
    heatmap = tf.reduce_sum(
        features * channel_weights[:, tf.newaxis, tf.newaxis, :],
        axis=-1
    )

    # ReLU and normalization
    heatmap = tf.nn.relu(heatmap[0])
    heatmap = heatmap / (tf.reduce_max(heatmap) + tf.keras.backend.epsilon())

    return heatmap.numpy(), predictions.numpy()

In [ ]:
os.makedirs(FIGURES_DIR, exist_ok=True)

# Convert the preprocessed sample to displayable RGB
original_image = display_image(sample_image[0])
if tf.is_tensor(original_image):
    original_image = original_image.numpy()

original_image = np.asarray(original_image)
original_image = np.clip(original_image, 0, 255).astype(np.uint8)

for name, (model, model_type) in gradcam_models.items():

    grad_model = build_gradcam_graph(model, model_type)

    heatmap, predictions = make_gradcam_heatmap(
        grad_model,
        sample_image
    )

    predicted_class = class_names[int(np.argmax(predictions[0]))]
    confidence = float(np.max(predictions[0]))

    # Resize heatmap to image dimensions
    heatmap_resized = tf.image.resize(
        heatmap[..., np.newaxis],
        (224, 224)
    ).numpy().squeeze()

    fig, axes = plt.subplots(1, 3, figsize=(15, 5))

    axes[0].imshow(original_image)
    axes[0].set_title("Original Image")

    axes[1].imshow(heatmap_resized, cmap="jet", vmin=0, vmax=1)
    axes[1].set_title("Grad-CAM Heatmap")

    axes[2].imshow(original_image)
    axes[2].imshow(
        heatmap_resized,
        cmap="jet",
        alpha=0.4,
        vmin=0,
        vmax=1
    )
    axes[2].set_title("Grad-CAM Overlay")

    for ax in axes:
        ax.axis("off")

    fig.suptitle(
        f"{name} | Predicted: {predicted_class} "
        f"| Confidence: {confidence:.4f}",
        fontsize=13
    )
    fig.tight_layout()

    filename = (
        name.lower()
        .replace(" + ", "_")
        .replace(" ", "_")
    )

    save_figure(fig, f"gradcam_{filename}.pdf")
    plt.show()
    plt.close(fig)

    print(
        f"Saved: gradcam_{filename}.pdf | "
        f"Prediction: {predicted_class} | "
        f"Confidence: {confidence:.4f}"
    )

In [ ]:
for filename in sorted(os.listdir(FIGURES_DIR)):
    if filename.startswith("gradcam_") and filename.endswith(".pdf"):
        filepath = os.path.join(FIGURES_DIR, filename)
        print(f"{filename}: {os.path.getsize(filepath) / 1024:.1f} KB")

In [ ]:

# SECTION 32: Combined Visualization Layout
# Input | Baseline Grad-CAM | Channel Grad-CAM |
# Spatial Attention Map | Channel + Spatial Grad-CAM

import os
import numpy as np
import tensorflow as tf
import matplotlib.pyplot as plt

os.makedirs(FIGURES_DIR, exist_ok=True)


# ---------------------------------------------------------
# 1. Grad-CAM heatmap
# ---------------------------------------------------------
def generate_gradcam(grad_model, image):
    with tf.GradientTape() as tape:
        features, predictions = grad_model(image, training=False)
        class_index = tf.argmax(predictions[0])
        class_score = predictions[:, class_index]

    gradients = tape.gradient(class_score, features)

    if gradients is None:
        raise ValueError("Grad-CAM gradients are None.")

    weights = tf.reduce_mean(gradients, axis=(1, 2))

    heatmap = tf.reduce_sum(
        features * weights[:, tf.newaxis, tf.newaxis, :],
        axis=-1
    )

    heatmap = tf.nn.relu(heatmap[0])
    heatmap = heatmap / (
        tf.reduce_max(heatmap) + tf.keras.backend.epsilon()
    )

    return heatmap.numpy(), predictions.numpy()


# ---------------------------------------------------------
# 2. Obtain the ACTUAL spatial attention weights
# ---------------------------------------------------------
def get_spatial_attention_map(image):
    inputs = tf.keras.Input(shape=(224, 224, 3))

    # Reuse trained layers from the spatial-attention model
    x = spatial_model.get_layer("functional_2")(inputs)

    attention_features = spatial_model.get_layer(
        "spatial_attention_pooling"
    )(x)

    attention_weights = spatial_model.get_layer(
        "conv2d"
    )(attention_features)

    attention_model = tf.keras.Model(
        inputs=inputs,
        outputs=attention_weights
    )

    attention_map = attention_model(image, training=False)[0, :, :, 0]
    attention_map = tf.clip_by_value(attention_map, 0.0, 1.0)

    return attention_map.numpy()


# ---------------------------------------------------------
# 3. Prepare the input image for visualization
# ---------------------------------------------------------
image_rgb = display_image(sample_image[0])

if tf.is_tensor(image_rgb):
    image_rgb = image_rgb.numpy()

image_rgb = np.asarray(image_rgb)

# Ensure the image can be displayed correctly
if image_rgb.max() <= 1.0:
    image_rgb = image_rgb * 255.0

image_rgb = np.clip(image_rgb, 0, 255).astype(np.uint8)


def resize_heatmap(heatmap):
    return tf.image.resize(
        heatmap[..., np.newaxis],
        (224, 224),
        method="bilinear"
    ).numpy().squeeze()


# ---------------------------------------------------------
# 4. Generate the three Grad-CAM maps
# ---------------------------------------------------------
baseline_grad_model = build_gradcam_graph(
    baseline_model, "baseline"
)

channel_grad_model = build_gradcam_graph(
    channel_model, "channel"
)

combined_grad_model = build_gradcam_graph(
    channel_spatial_model, "channel_spatial"
)

baseline_heatmap, baseline_predictions = generate_gradcam(
    baseline_grad_model, sample_image
)

channel_heatmap, channel_predictions = generate_gradcam(
    channel_grad_model, sample_image
)

combined_heatmap, combined_predictions = generate_gradcam(
    combined_grad_model, sample_image
)

# Generate the real spatial-attention weights
spatial_attention = get_spatial_attention_map(sample_image)

# Resize all maps to match the input image
baseline_heatmap = resize_heatmap(baseline_heatmap)
channel_heatmap = resize_heatmap(channel_heatmap)
combined_heatmap = resize_heatmap(combined_heatmap)
spatial_attention = resize_heatmap(spatial_attention)


# ---------------------------------------------------------
# 5. Create the exact five-panel layout from Section 32
# ---------------------------------------------------------
fig, axes = plt.subplots(1, 5, figsize=(24, 5.5))

# Panel 1: Input
axes[0].imshow(image_rgb)
axes[0].set_title("Input Image", fontsize=12)

# Panel 2: Baseline Grad-CAM overlay
axes[1].imshow(image_rgb)
axes[1].imshow(
    baseline_heatmap, cmap="jet", alpha=0.45,
    vmin=0, vmax=1
)
axes[1].set_title("Baseline\nGrad-CAM", fontsize=12)

# Panel 3: Channel Grad-CAM overlay
axes[2].imshow(image_rgb)
axes[2].imshow(
    channel_heatmap, cmap="jet", alpha=0.45,
    vmin=0, vmax=1
)
axes[2].set_title("Channel\nGrad-CAM", fontsize=12)

# Panel 4: Actual spatial-attention map overlay
axes[3].imshow(image_rgb)
axes[3].imshow(
    spatial_attention, cmap="jet", alpha=0.45,
    vmin=0, vmax=1
)
axes[3].set_title("Spatial\nAttention Map", fontsize=12)

# Panel 5: Channel + Spatial Grad-CAM overlay
axes[4].imshow(image_rgb)
axes[4].imshow(
    combined_heatmap, cmap="jet", alpha=0.45,
    vmin=0, vmax=1
)
axes[4].set_title("Channel + Spatial\nGrad-CAM", fontsize=12)

for ax in axes:
    ax.axis("off")

# Show the ground-truth and model predictions
true_label = int(np.asarray(sample_label).reshape(-1)[0])

fig.suptitle(
    f"Section 32: Attention Visualization Comparison"
    f"\nGround Truth: {class_names[true_label]}",
    fontsize=15,
    y=1.05
)

fig.tight_layout()

# Save as PDF at 600 DPI
output_path = os.path.join(
    FIGURES_DIR,
    "section32_combined_visualization.pdf"
)

fig.savefig(
    output_path,
    format="pdf",
    dpi=600,
    bbox_inches="tight"
)

plt.show()
plt.close(fig)

print("Section 32 figure saved successfully:")
print(output_path)

print("\nPredictions:")
for name, predictions in [
    ("Baseline", baseline_predictions),
    ("Channel", channel_predictions),
    ("Spatial", spatial_model(sample_image, training=False).numpy()),
    ("Channel + Spatial", combined_predictions)
]:
    pred_index = int(np.argmax(predictions[0]))
    confidence = float(np.max(predictions[0]))

    print(
        f"{name}: {class_names[pred_index]} "
        f"(confidence: {confidence:.4f})"
    )


In [ ]:

# OPTIONAL ATTENTION COMPARISON
# Input | Baseline | SE | ECA | CBAM | Self-Attention

import os
import numpy as np
import tensorflow as tf
import matplotlib.pyplot as plt

os.makedirs(FIGURES_DIR, exist_ok=True)


def input_gradient_saliency(model, image):
    """Compute input-gradient saliency for an arbitrary classifier."""
    image_tensor = tf.convert_to_tensor(image, dtype=tf.float32)

    with tf.GradientTape() as tape:
        tape.watch(image_tensor)
        predictions = model(image_tensor, training=False)
        class_index = tf.argmax(predictions[0])
        score = predictions[:, class_index]

    gradients = tape.gradient(score, image_tensor)

    if gradients is None:
        raise ValueError("Could not calculate input gradients.")

    # Absolute gradient, aggregated across RGB channels
    saliency = tf.reduce_max(tf.abs(gradients), axis=-1)[0]

    # Normalize to [0, 1]
    saliency = saliency / (
        tf.reduce_max(saliency) + tf.keras.backend.epsilon()
    )

    return saliency.numpy(), predictions.numpy()


# Use the same sample image for every model
image_rgb = display_image(sample_image[0])

if tf.is_tensor(image_rgb):
    image_rgb = image_rgb.numpy()

image_rgb = np.asarray(image_rgb)

if image_rgb.max() <= 1.0:
    image_rgb = image_rgb * 255.0

image_rgb = np.clip(image_rgb, 0, 255).astype(np.uint8)


# Models for the optional comparison
optional_models = {
    "SE": se_model,
    "ECA": eca_model,
    "CBAM": cbam_model,
    "Self-Attention": self_attention_model,
}


# Baseline Grad-CAM
baseline_grad_model = build_gradcam_graph(
    baseline_model, "baseline"
)

baseline_heatmap, baseline_predictions = generate_gradcam(
    baseline_grad_model, sample_image
)

baseline_heatmap = tf.image.resize(
    baseline_heatmap[..., np.newaxis],
    (224, 224)
).numpy().squeeze()


# Create a single six-panel figure
fig, axes = plt.subplots(1, 6, figsize=(25, 5.5))

axes[0].imshow(image_rgb)
axes[0].set_title("Input Image", fontsize=12)

axes[1].imshow(image_rgb)
axes[1].imshow(
    baseline_heatmap, cmap="jet", alpha=0.45,
    vmin=0, vmax=1
)
axes[1].set_title("Baseline\nGrad-CAM", fontsize=12)

# Generate saliency overlays for the optional models
for ax, (name, model) in zip(axes[2:], optional_models.items()):
    saliency, predictions = input_gradient_saliency(
        model, sample_image
    )

    saliency = tf.image.resize(
        saliency[..., np.newaxis],
        (224, 224)
    ).numpy().squeeze()

    ax.imshow(image_rgb)
    ax.imshow(
        saliency, cmap="jet", alpha=0.45,
        vmin=0, vmax=1
    )
    ax.set_title(
        f"{name}\nInput-Gradient Saliency",
        fontsize=11
    )

    predicted_class = int(np.argmax(predictions[0]))
    confidence = float(np.max(predictions[0]))

    print(
        f"{name}: {class_names[predicted_class]} "
        f"(confidence: {confidence:.4f})"
    )

for ax in axes:
    ax.axis("off")

fig.suptitle(
    "Optional Attention Mechanism Comparison",
    fontsize=15,
    y=1.04
)

fig.tight_layout()

output_path = os.path.join(
    FIGURES_DIR,
    "optional_attention_comparison.pdf"
)

fig.savefig(
    output_path,
    format="pdf",
    dpi=600,
    bbox_inches="tight"
)

plt.show()
plt.close(fig)

print("\nSaved:", output_path)


In [ ]:
import shutil
from IPython.display import FileLink, display

# Folder containing your generated figures
figures_path = "/kaggle/working/Figures"

# Create a ZIP archive in the output directory
zip_path = shutil.make_archive(
    "/kaggle/working/Figures",
    "zip",
    root_dir="/kaggle/working",
    base_dir="Figures"
)

print("Figures folder compressed successfully!")
print("ZIP file:", zip_path)

# Display a clickable download link
display(FileLink(zip_path))